# Local Nonlinear Support Verification

This notebook is the first safe nonlinear verification workflow for the local support assembly idea. It intentionally uses NGSolve's global FESpace and automatic linearization, then restricts integration to each patch's `support_elements` with `dx(definedonelements=...)`.

This is not yet a true local-size nonlinear C++ kernel. The purpose is to verify that the same Python-built `LocalSupportPatch` used in the linear matrix/vector workflow also reproduces the nonlinear residual entries and Jacobian core-core block after extraction.

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import unit_square
import numpy as np

from partition.metis import metis_partition_from_fes
from utils.patches import build_support_patches, print_patch_summary, draw_patch

import myassembling
print(myassembling)
print(myassembling.__file__)
print(dir(myassembling))

## Mesh, Space, and Support Patches

The support patch definition is unchanged from the linear workflow. Starting from `core_elements`, Python builds `core_dofs`, then takes all elements whose DoF lists touch those core DoFs. These are the `support_elements`.

In [ ]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.12))
order = 1
fes = H1(mesh, order=order, dirichlet="left|right|bottom|top")

core_partition, overlapping_partition, cutcount = metis_partition_from_fes(
    fes,
    nparts=3,
    overlap_width=1,
    free_dofs_only=False,
)

partition = overlapping_partition
patches = build_support_patches(fes, partition)

print("mesh.ne =", mesh.ne)
print("fes.ndof =", fes.ndof)
print("METIS cutcount =", cutcount)
print("core element counts =", [len(part) for part in partition])

for i, patch in enumerate(patches):
    print(f"\npatch {i}")
    print_patch_summary(patch)

In [ ]:
for i in range(len(partition)):
    draw_patch(mesh, patches[i], name="nonlinear support patch 0")

## Nonlinear Form

We use the nonlinear weak form

`F(u; v) = int grad(u) . grad(v) dx + int (1/3) u^3 v dx - int 10 v dx`.

`BilinearForm.Apply(gfu.vec, residual)` evaluates the nonlinear operator at the current state. `BilinearForm.AssembleLinearization(gfu.vec)` asks NGSolve to assemble the Jacobian at the same state, and `a.mat` then stores that Jacobian.

In [ ]:
u, v = fes.TnT()

def add_nonlinear_terms(a, dx_measure):
    a += (
        grad(u) * grad(v)
        + (1.0 / 3.0) * u**3 * v
        - 10.0 * v
    ) * dx_measure
    return a

a_global = BilinearForm(fes)
add_nonlinear_terms(a_global, dx)

gfu = GridFunction(fes, name="u_current")
gfu.Set((x * (1 - x))**2 * (y * (1 - y))**2)
Draw(gfu, mesh, "u_current")

In [ ]:
res_global = gfu.vec.CreateVector()
a_global.Apply(gfu.vec, res_global)

a_global.AssembleLinearization(gfu.vec)
J_global = a_global.mat

## Restricted Support Forms

For each patch, we build a global-size NGSolve form whose integrals are restricted to `patch.support_elements`. The residual and Jacobian are therefore global-size objects, but only the support elements contribute.

This is a correctness prototype. A future `src/myassembling_nonlinear.cpp` kernel may assemble true local-size `F_support` and `J_support` using `support_dofs` as local numbering.

In [ ]:
def element_bitarray(mesh, elements):
    ba = BitArray(mesh.ne)
    ba[:] = False
    for elnr in elements:
        ba[int(elnr)] = True
    return ba

def dense_submatrix(A, rows, cols):
    return np.array([[A[i, j] for j in cols] for i in rows], dtype=float)

def vector_entries(vec, rows):
    return np.array([vec[i] for i in rows], dtype=float)

In [ ]:
restricted_results = []

for patch in patches:
    ba = element_bitarray(mesh, patch.support_elements)

    a_patch = BilinearForm(fes)
    add_nonlinear_terms(a_patch, dx(definedonelements=ba))

    res_patch = gfu.vec.CreateVector()
    a_patch.Apply(gfu.vec, res_patch)

    a_patch.AssembleLinearization(gfu.vec)
    J_patch = a_patch.mat

    restricted_results.append((res_patch, J_patch))

## Core Extraction Verification

Since `support_elements` contains every element touching `core_dofs`, all contributions to the nonlinear residual entries on `core_dofs` are present in the restricted patch form. The same support closure also contains all element contributions needed for the Jacobian block on `core_dofs x core_dofs`.

We verify only these extracted quantities:

`F_patch[core_dofs] == F_global[core_dofs]`

`J_patch[core_dofs, core_dofs] == J_global[core_dofs, core_dofs]`

In [ ]:
for i, (patch, (res_patch, J_patch)) in enumerate(zip(patches, restricted_results)):
    F_core_from_patch = vector_entries(res_patch, patch.core_dofs)
    F_core_from_global = vector_entries(res_global, patch.core_dofs)
    err_F = np.linalg.norm(F_core_from_patch - F_core_from_global, ord=np.inf)

    J_core_from_patch = dense_submatrix(
        J_patch,
        patch.core_dofs,
        patch.core_dofs,
    )
    J_core_from_global = dense_submatrix(
        J_global,
        patch.core_dofs,
        patch.core_dofs,
    )
    err_J = np.linalg.norm(J_core_from_patch - J_core_from_global, ord=np.inf)

    print(f"patch {i}:")
    print(f"  ||F_patch[core] - F_global[core]||_inf = {err_F:.3e}")
    print(f"  ||J_patch[core,core] - J_global[core,core]||_inf = {err_J:.3e}")

    assert err_F < 1e-10
    assert err_J < 1e-10

The verified objects above are global-size restricted residuals and Jacobians. They validate the support-closure idea for nonlinear residuals and automatic Jacobians without changing the stable linear C++ local assembly path.